In [104]:
from utils.utils import load_all_games_csv, basic_win_prob_for_et, get_teams
from scipy.special import expit
from sklearn.metrics import log_loss, accuracy_score
import numpy as np
import pandas as pd
from typing import Tuple
from elos.elo_tracker import EloTracker

# Model vs 538

This will compare my best model to the 538 baseline, for the 2024 season.

## Get 2024 Season Games

In [105]:
games = load_all_games_csv('../data/gameinfo_cleaned.csv')
games = games[games['season']==2024]
games.head()

/Users/lancehendricks/Documents/College Coding/ML/Elo Ratings/analysis/src/utils/utils.py:27: DtypeWarning: Columns (10,11,13,17,19,21,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  all_games = pd.read_csv(filename)


,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,...,homerestdays,visrestdays,homepitcherrgs,vispitcherrgs,hometeamrgs,visteamrgs,homepitcherminusteamrgs,vispitcherminusteamrgs,homelastkwinpct,vislastkwinpct
gid,,,,,,,,,,,,,,,,,,,,,
SDN202403200,LAN,SDN,SEO01,20240320,0.0,7:05PM,night,9.0,2.0,True,...,171.0,161.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
LAN202403210,SDN,LAN,SEO01,20240321,0.0,7:05PM,night,9.0,2.0,True,...,1.0,1.0,39.0,39.0,54.9,53.9,-15.9,-14.9,1.0,0.0
ARI202403280,COL,ARI,PHO01,20240328,0.0,7:10PM,night,9.0,2.0,True,...,148.0,179.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
BAL202403280,ANA,BAL,BAL12,20240328,0.0,3:05PM,day,9.0,2.0,True,...,170.0,179.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
CHA202403280,DET,CHA,CHI12,20240328,0.0,3:10PM,day,9.0,2.0,True,...,179.0,179.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN


In [106]:
# Max rest days
games['visrestdays'] = games['visrestdays'].apply(lambda x: min(3,x))
games['homerestdays'] = games['homerestdays'].apply(lambda x: min(3,x))


In [107]:
# Take cube root of distance traveled
games['homedistancetraveled'] = games['homedistancetraveled']**(1/3)
games['visdistancetraveled'] = games['visdistancetraveled']**(1/3)

In [108]:
# Take square root of margin of victory
games['marginofvictory'] = np.sqrt(games['marginofvictory'])
games.head()

,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,...,homerestdays,visrestdays,homepitcherrgs,vispitcherrgs,hometeamrgs,visteamrgs,homepitcherminusteamrgs,vispitcherminusteamrgs,homelastkwinpct,vislastkwinpct
gid,,,,,,,,,,,,,,,,,,,,,
SDN202403200,LAN,SDN,SEO01,20240320,0.0,7:05PM,night,9.0,2.0,True,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
LAN202403210,SDN,LAN,SEO01,20240321,0.0,7:05PM,night,9.0,2.0,True,...,1.0,1.0,39.0,39.0,54.9,53.9,-15.9,-14.9,1.0,0.0
ARI202403280,COL,ARI,PHO01,20240328,0.0,7:10PM,night,9.0,2.0,True,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
BAL202403280,ANA,BAL,BAL12,20240328,0.0,3:05PM,day,9.0,2.0,True,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
CHA202403280,DET,CHA,CHI12,20240328,0.0,3:10PM,day,9.0,2.0,True,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN


## Model Evaluation Functions

In [109]:

def add_elos_to_games_df(games_df: pd.DataFrame, elo_prob_func=basic_win_prob_for_et, K: float = 3, use_margin_of_victory: bool = False) -> pd.DataFrame:
    """Returns a version of games_df with columns 'homeelo' and 'viselo' added, which
    are calculated in part with the Elo probability function, elo_prob_func.
    
    Removes any na rows for important features at the end.
    
    Args:
        games_df (pd.DataFrame): Table whose rows are chronologically ordered game box scores,
                including columns 'hometeam' for the home team, 'visteam' for the away team, and
                'homewon' which is True if home won and False otherwise. Each game in game_df must take
                place after the games that have already been logged for the given teams it includes.
                Must be indexed by a game id column 'gid'.
        elo_prob_func (function): Function that takes in a home elo, away elo, and game information
            (i.e. row of box scores dataframe) and produces the probability of the home team winning.
        K (float): The K factor, controlling how sensitive each Elo update should be.
        use_margin_of_victory (bool): If True, incorporates margin of victory in the Elo update, where
                higher margins result in larger updates. It is included as an additional variable multiplied by K.
    """
    games_df = games_df.copy() # Don't modify original
    
    teams = get_teams(games_df)
    
    # First, get all Elo ratings
    et = EloTracker(teams, elo_prob_func=elo_prob_func, K=K, use_margin_of_victory=use_margin_of_victory)
    
    et.add_history(games_df)
    
    # Add raw pre-game Elo Ratings
    games_df['homeelo'] = [0.0] * len(games_df)
    games_df['viselo'] = [0.0] * len(games_df)
    
    home_elos = {}
    vis_elos = {}

    for team in teams:
        #print(len(et.elos_map[team]))
        for game in et.elos_map[team]:
            gid = game[0]
            elo = game[2] # Before update
            #print(elo)
        
            if games_df.loc[gid,'hometeam'] == team:
                home_elos[gid] = elo
            else:
                vis_elos[gid] = elo
                
    games_df['homeelo'] = games_df.index.map(home_elos)
    games_df['viselo'] = games_df.index.map(vis_elos)
                
    # Drop rows with na travel distance, rest or pitcher info
    games_df = games_df.dropna(subset=['homedistancetraveled', 'visdistancetraveled', 'homerestdays', 'visrestdays', 'homepitcherminusteamrgs', 'vispitcherminusteamrgs']).copy()
    
    #print(games_df[games_df['homeelo'].isna() | games_df['viselo'].isna()])
                
    return games_df


In [110]:
def evaluate_elo_prob_func(games_df: pd.DataFrame, elo_prob_func=basic_win_prob_for_et, K: float = 3, use_margin_of_victory: bool = False, skip_first_n: int=0) -> Tuple[float, float]:
    """Evaluates how well the given function to calculate Elo probabilties does on games_df,
    producing binary cross entropy and accuracy.
    
    Args:
        games_df (pd.DataFrame): Table whose rows are chronologically ordered game box scores,
                including columns 'hometeam' for the home team, 'visteam' for the away team, and
                'homewon' which is True if home won and False otherwise. Each game in game_df must take
                place after the games that have already been logged for the given teams it includes.
                Must be indexed by a game id column 'gid'.
        elo_prob_func (function): Function that takes in a home elo, away elo, and game information
            (i.e. row of box scores dataframe) and produces the probability of the home team winning.
        K (float): The K factor, controlling how sensitive each Elo update should be.
        use_margin_of_victory (bool): If True, incorporates margin of victory in the Elo update, where
                higher margins result in larger updates. It is included as an additional variable multiplied by K.
        skip_first_n (int): The first skip_first_n games for each team will not be considered when computing the accuracy or
            cross entropy metrics, to allow time for the ratings to adjust to performance.
    """
    
    # Add elos
    games_df = add_elos_to_games_df(games_df, elo_prob_func, K=K, use_margin_of_victory=use_margin_of_victory)
    
    # Filter out games where the team hasn't more than played skip_first_n
    games_df = games_df[(games_df['hometeamgamecount'] > skip_first_n) & (games_df['visteamgamecount'] > skip_first_n)]
        
    games_df['homewinprob'] = games_df.apply(lambda game: elo_prob_func(game['homeelo'], game['viselo'], game), axis=1)
    
    bce = log_loss(games_df['homewon'], games_df['homewinprob'])
    accuracy = accuracy_score(games_df['homewon'], round(games_df['homewinprob']))
    
    return bce, accuracy

## Evaluate my Model

In [111]:
def p(X,w):
    """Vector form Elo pdf for a tabular input."""
    z = X @ w
    return expit((-np.log(10) / 400) * z)

In [126]:
w = np.array([[ 1.        ],
       [27.1440666 ],
       [ 4.81476328],
       [-0.30523283],
       [ 1.58004319]])

In [113]:
def predict_lr(home_elo, away_elo, game, w):
    
    elo_diff = away_elo - home_elo
    rest_day_diff = game['visrestdays'] - game['homerestdays']
    rest_day_diff = rest_day_diff if not np.isnan(rest_day_diff) else 0
    
    travel_diff = game['visdistancetraveled'] - game['homedistancetraveled']
    travel_diff = travel_diff if not np.isnan(travel_diff) else 0
    
    home_adv_diff = 0 - 1
    
    pitcher_diff = game['vispitcherminusteamrgs'] - game['homepitcherminusteamrgs']
    pitcher_diff = pitcher_diff if not np.isnan(pitcher_diff) else 0
    
    x = np.array([elo_diff, home_adv_diff, rest_day_diff, travel_diff, pitcher_diff]).reshape(-1,1)
     
    return p(x.T, w).item()

In [127]:
bce, accuracy = evaluate_elo_prob_func(games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w), K=3, use_margin_of_victory=True, skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6829313083151032
Accuracy: 0.5542071197411004


## Evaluate 538 Model

In [115]:
w_538 = np.array([[ 1.        ],
       [24 ],
       [ 2.3],
       [-0.31],
       [ 4.7]])

In [129]:
bce, accuracy = evaluate_elo_prob_func(games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w_538), K=3, use_margin_of_victory=False, skip_first_n=0)
print(f"538 BCE: {bce}")
print(f"538 Accuracy: {accuracy}")

538 BCE: 0.6888037824221744
538 Accuracy: 0.5469255663430421
